In [ ]:
# ====================================================================
# CELL 1: INSTALL EVALUATION DEPENDENCIES + GPU DETECTION
# ====================================================================
import subprocess, sys, time
from pathlib import Path

WALL_START = time.time()

def pip(*args, optional=False):
    # pip install wrapper; optional=True warns instead of raising.
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *args],
    )
    if r.returncode != 0:
        if optional:
            print(f'  warning: optional install failed: {" ".join(args)}')
            return False
        r.check_returncode()
    return True

# -- 1. Core packages (already on Kaggle; ensure up-to-date) ----------------
print('[1/3] Installing pandas / matplotlib / seaborn ...')
pip('pandas>=1.3', 'matplotlib', 'seaborn')

# -- 2. motmetrics (IDF1 / MOTA / MOTP) — optional -------------------------
# Install WITHOUT --no-deps so lap (required dep) gets pulled in.
# Do NOT reinstall scipy/numpy — keep system versions intact.
# The actual motmetrics import happens in a subprocess (cell 6) to avoid
# any in-kernel numpy/scipy version confusion.
print('[2/3] Installing motmetrics ...')
_mm_ok = pip('motmetrics>=1.4.0', optional=True)

# -- 3. trackeval (HOTA) — optional ----------------------------------------
print('[3/3] Installing trackeval ...')
_te_ok = pip('trackeval', '--no-deps', optional=True)
if not _te_ok:
    _te_ok = pip('trackeval', optional=True)

# -- GPU / device check -----------------------------------------------------
import torch
_gpu   = torch.cuda.is_available()
DEVICE = torch.device('cuda' if _gpu else 'cpu')

if _gpu:
    _props = torch.cuda.get_device_properties(0)
    print(f'\nGPU detected : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM       : {_props.total_memory / 1e9:.1f} GB')
    print(f'  CUDA       : {torch.version.cuda}')
else:
    print('No CUDA GPU -- ReID will run on CPU.')

import numpy as np
import scipy as _sp
print(f'\n  numpy  : {np.__version__}')
print(f'  scipy  : {_sp.__version__}')
print(f'  motmetrics available : {_mm_ok}')
print(f'  trackeval  available : {_te_ok}')
print(f'\nCell 1 done.  Device = {DEVICE}  ({time.time()-WALL_START:.1f}s)')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 2: CONFIGURATION + KAGGLE AUTH
# ══════════════════════════════════════════════════════════════════
import os
from pathlib import Path

# -- Kaggle Secrets auth ---------------------------------------------------
if 'KAGGLE_API_TOKEN' not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        _sec = UserSecretsClient()
        os.environ['KAGGLE_API_TOKEN'] = _sec.get_secret('KAGGLE_API_TOKEN')
        print('✅ KAGGLE_API_TOKEN loaded from Kaggle Secrets.')
    except Exception:
        print('⚠️  KAGGLE_API_TOKEN not found.')
        print('   Add it under Notebook Settings → Secrets to enable auto-push.')
else:
    print('✅ KAGGLE_API_TOKEN found in environment.')

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'user')
print(f'Kaggle username: {KAGGLE_USERNAME}')

# -- Working paths ---------------------------------------------------------
WORK        = Path('/kaggle/working')
EVAL_DIR    = WORK / 'stage5_eval'
PRED_DIR    = EVAL_DIR / 'predictions'
GT_DIR      = EVAL_DIR / 'gt'
RESULTS_DIR = EVAL_DIR / 'results'
for d in [EVAL_DIR, PRED_DIR, GT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -- Hardcoded AIC22 dataset root ------------------------------------------
AIC22_ROOT = Path('/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2')
if not AIC22_ROOT.exists():
    print(f'⚠️  AIC22 dataset not found at {AIC22_ROOT}')
    AIC22_ROOT = None
else:
    print(f'AIC22 root auto-detected: {AIC22_ROOT}')

print(f'\nAIC22 dataset root : {AIC22_ROOT}')
print(f'Eval directory     : {EVAL_DIR}')

print('\n/kaggle/input contents:')
for p in sorted(Path('/kaggle/input').iterdir()):
    print(f'  {p}')
print('=' * 60)


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 3: SELECTIVE EXTRACTION + LOAD DATA
# !! Only extracts the 3 files actually needed -- NOT the full zip !!
# ══════════════════════════════════════════════════════════════════
import json, zipfile
import shutil as _shutil
import numpy as np
from pathlib import Path

print('=' * 60)
print('CELL 3: SELECTIVE EXTRACTION + LOAD DATA')
print('=' * 60)

free_gb = _shutil.disk_usage('/kaggle/working').free / 1e9
print(f'Disk free at start : {free_gb:.1f} GB')

EXTRACT_ROOT = WORK / 'all_unzipped'
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_FILES = {
    'global_tracklets.json',
    'tracklets.json',
    'reid_features_improved.npz',
    'reid_features.npz',
}

# -- Hardcoded zip paths ---------------------------------------------------
HARDCODED_ZIPS = [
    Path('/kaggle/input/notebooks/seiftamerheakal/gp-stage-4/_output_.zip'),
    Path('/kaggle/input/notebooks/gumfreddy/gp-stage-1b-detection-tracking-deepocsort/_output_.zip'),
]
all_zips = [p for p in HARDCODED_ZIPS if p.exists()]

EXTRACT_DIRS = []

if not all_zips:
    print('WARNING: No _output_.zip found at expected paths:')
    for p in HARDCODED_ZIPS:
        print(f'   {p}')
    print('   Continuing -- metrics needing Stage 4 data will be skipped.')
else:
    print(f'Found {len(all_zips)} kernel output zip(s):')
    for i, zp in enumerate(all_zips):
        size_gb = zp.stat().st_size / 1e9
        print(f'\n  [{i}] {zp}  ({size_gb:.1f} GB)')
        dest = EXTRACT_ROOT / f'zip_{i}'
        dest.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zp, 'r') as zf:
            found_any = False
            for name in zf.namelist():
                basename = Path(name).name
                if basename not in TARGET_FILES:
                    continue
                out_path = dest / basename
                if out_path.exists():
                    print(f'    -> Already cached : {basename}')
                    found_any = True
                    continue
                sz_mb = zf.getinfo(name).file_size / 1e6
                print(f'    -> Extracting : {name}  ({sz_mb:.1f} MB uncompressed)', flush=True)
                data = zf.read(name)
                out_path.write_bytes(data)
                del data
                print(f'      ✓ saved', flush=True)
                found_any = True
            if not found_any:
                print('    No target files found in this zip.')

        free_now = _shutil.disk_usage('/kaggle/working').free / 1e9
        print(f'    Disk free after zip {i} : {free_now:.1f} GB')
        EXTRACT_DIRS.append(dest)

# -- Locate key files ------------------------------------------------------
def find_file_multi(dirs, pattern):
    for d in dirs:
        results = list(d.rglob(pattern))
        if results:
            return results[0]
    return None

def find_tracklets_with_bboxes(dirs):
    candidates = []
    for d in dirs:
        for f in d.rglob('tracklets.json'):
            candidates.append(f)
    if not candidates:
        return None, False
    for f in candidates:
        try:
            with open(f) as fh:
                sample = json.load(fh)
            if any('bboxes' in v for v in list(sample.values())[:5]):
                return f, True
        except Exception:
            pass
    return candidates[0], False

GLOBAL_TRACKLETS_JSON             = find_file_multi(EXTRACT_DIRS, 'global_tracklets.json')
FEATURES_NPZ                      = (find_file_multi(EXTRACT_DIRS, 'reid_features_improved.npz') or
                                      find_file_multi(EXTRACT_DIRS, 'reid_features.npz'))
STAGE1_TRACKLETS_JSON, has_bboxes = find_tracklets_with_bboxes(EXTRACT_DIRS)
CROPS_ROOT                        = WORK / 'crops'

print('\nFiles found:')
for label, path in [
    ('global_tracklets.json', GLOBAL_TRACKLETS_JSON),
    ('reid_features.npz    ', FEATURES_NPZ),
    ('tracklets.json (S1)  ', STAGE1_TRACKLETS_JSON),
]:
    status = str(path) if path else 'NOT FOUND'
    print(f'  {label}: {status}')

# -- Load ------------------------------------------------------------------
print(f'\nLoading data  (bbox data: {has_bboxes}) ...')

if GLOBAL_TRACKLETS_JSON is None:
    print('WARNING: global_tracklets.json not found.')
    global_tracklets = {}
else:
    with open(GLOBAL_TRACKLETS_JSON) as f:
        global_tracklets = json.load(f)
print(f'  global_tracklets : {len(global_tracklets)} global IDs')

stage1_tracklets = {}
if STAGE1_TRACKLETS_JSON and STAGE1_TRACKLETS_JSON.exists():
    with open(STAGE1_TRACKLETS_JSON) as f:
        stage1_tracklets = json.load(f)
    print(f'  stage1_tracklets : {len(stage1_tracklets)} tracks  (bbox data: {has_bboxes})')
    if not has_bboxes:
        print('  ⚠️  No bbox data in tracklets.json -- MOT metrics will be skipped.')
else:
    print('  stage1_tracklets : not found -- MOT metrics will be skipped.')

features  = None
reid_meta = []
N, D      = 0, 0

if FEATURES_NPZ and FEATURES_NPZ.exists():
    data_npz  = np.load(FEATURES_NPZ, allow_pickle=True)
    features  = data_npz['features'].astype(np.float32)
    meta_raw  = data_npz['metadata']
    reid_meta = (meta_raw.item()
                 if (isinstance(meta_raw, np.ndarray) and meta_raw.ndim == 0)
                 else list(meta_raw))
    norms     = np.linalg.norm(features, axis=1, keepdims=True)
    features  = features / np.maximum(norms, 1e-8)
    N, D      = features.shape
    print(f'  ReID features    : {features.shape}  (L2-normalised)')
else:
    print('  ReID features    : NOT FOUND -- mAP/CMC metrics will be skipped.')

print('\n' + '=' * 60)
print(f'✅ Data loading complete.  N={N} tracklets, D={D} dims')
print('=' * 60)


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 4: AUTO-DETECT AIC22 GROUND TRUTH
# Dataset: thanhnguyenle/data-aicity-2023-track-2
# Structure:
#   train/S01/c001/gt/gt.txt  → per-camera MOT GT (IDF1/MOTA)
#   test_tracks.json          → vehicle identity GT (ReID mAP/CMC)
#   test_queries.json         → ReID query set
# ══════════════════════════════════════════════════════════════════
import json, re
from collections import defaultdict

print("=" * 60)
print("SCANNING AIC22 GROUND TRUTH")
print("=" * 60)

GT_MODE   = "none"
gt_mot    = {}   # {(scenario, camera): {frame_id: [(track_id,x,y,w,h), ...]}}
gt_reid   = {}   # {vehicle_id: [crop_path, ...]}

# 1) MOT gt.txt — lives at train/Sxx/cxxx/gt/gt.txt
gt_txt_files = []
if AIC22_ROOT:
    gt_txt_files = list(AIC22_ROOT.rglob("gt.txt"))
    print(f"  Found {len(gt_txt_files)} gt.txt file(s)")
    for gf in gt_txt_files[:3]:
        print(f"    {gf}")

if gt_txt_files:
    GT_MODE = "mot"
    for gt_file in gt_txt_files:
        m = re.search(r'(S\d+)[/\\](c\d+)', str(gt_file))
        scenario = m.group(1) if m else "S00"
        camera   = m.group(2) if m else "c000"
        frames   = defaultdict(list)
        with open(gt_file) as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) < 6:
                    continue
                fid, tid = int(parts[0]), int(parts[1])
                x, y, w, h = float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])
                frames[fid].append((tid, x, y, w, h))
        gt_mot[(scenario, camera)] = dict(frames)
    total_f = sum(len(v) for v in gt_mot.values())
    print(f"\n✅ MOT GT: {len(gt_mot)} cameras,  {total_f} annotated frame-sets")

# 2) ReID GT — test_tracks.json or train-tracks.json
reid_files = []
if AIC22_ROOT:
    reid_files = (
        list(AIC22_ROOT.rglob("test_tracks.json")) +
        list(AIC22_ROOT.rglob("train-tracks.json")) +
        list(AIC22_ROOT.rglob("train_tracks.json"))
    )
if reid_files:
    with open(reid_files[0]) as f:
        raw_reid = json.load(f)
    # Normalise: expect {vehicle_id: [crop_path, ...]}
    # test_tracks.json may have {vehicle_id: {"cameras":..., "frames":...}}
    if raw_reid and isinstance(next(iter(raw_reid.values())), list):
        gt_reid = raw_reid   # already in expected format
    else:
        # Convert dict-of-dicts → skip (can't map to reid_meta crop_paths)
        # Will fall back to proxy GT; at least MOT GT is active
        gt_reid = {}
        print(f"  ReID GT format not crop-path list — skipping ReID official GT")
    if gt_reid:
        total_crops = sum(len(v) for v in gt_reid.values())
        print(f"✅ ReID GT: {len(gt_reid)} vehicles,  {total_crops} crop annotations  [{reid_files[0].name}]")
        if GT_MODE == "none":
            GT_MODE = "reid"

print(f"\n{'=':=<60}")
print(f"  GT MODE: {GT_MODE.upper()}")
if GT_MODE == "none":
    print("  ⚠️  No AIC22 GT found.")
    print("     Attach dataset: thanhnguyenle/data-aicity-2023-track-2")
    print("     Proceeding with camera-consistency proxy GT (approximate).")
print("=" * 60)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 5: BUILD MOT PREDICTION FILES
# Converts global_tracklets.json + stage1 bboxes → per-camera
# prediction files in MOT Challenge format:
#   frame, global_id, x, y, w, h, conf, -1, -1, -1
# ══════════════════════════════════════════════════════════════════
from collections import defaultdict
from tqdm.auto import tqdm

print("=" * 60)
print("BUILDING MOT PREDICTION FILES")
print("=" * 60)

# local_tracklet_id → global_id lookup
local_to_global = {}
for gid_str, cam_tracks in global_tracklets.items():
    gid = int(gid_str)
    tracks_list = cam_tracks if isinstance(cam_tracks, list) else list(cam_tracks.values())
    for ct in tracks_list:
        lid = ct.get("local_tracklet_id")
        if lid:
            local_to_global[lid] = gid

print(f"local→global mapping: {len(local_to_global)} entries")

# Build per-camera prediction dicts
pred_data = defaultdict(lambda: defaultdict(list))
# pred_data[(scenario, camera)][frame] = [(global_id, x, y, w, h, conf)]

if has_bboxes:
    for local_id, track_info in tqdm(stage1_tracklets.items(), desc="Building preds"):
        global_id = local_to_global.get(local_id)
        if global_id is None:
            # Assign a unique singleton ID (won't match any GT cross-camera pair)
            global_id = 1_000_000 + int(local_id.split("_")[-1])

        scenario = track_info.get("scenario", "S00")
        camera   = track_info.get("camera_id", "c000")
        cam_key  = (scenario, camera)

        for bbox in track_info.get("bboxes", []):
            frame = bbox["frame"] + 1  # 1-indexed (MOT format)
            pred_data[cam_key][frame].append((
                global_id,
                bbox["x1"], bbox["y1"],
                bbox["w"],  bbox["h"],
                bbox["conf"]
            ))

    # Write prediction files
    n_files = 0
    total_det = 0
    for (scenario, camera), frame_dict in sorted(pred_data.items()):
        cam_dir = PRED_DIR / scenario / camera
        cam_dir.mkdir(parents=True, exist_ok=True)
        with open(cam_dir / "pred.txt", "w") as f:
            for frame in sorted(frame_dict.keys()):
                for gid, x, y, w, h, conf in frame_dict[frame]:
                    f.write(f"{frame},{gid},{x:.1f},{y:.1f},{w:.1f},{h:.1f},{conf:.4f},-1,-1,-1\n")
            total_det += sum(len(v) for v in frame_dict.values())
        n_files += 1

    print(f"\n✅ Saved {n_files} prediction files  ({total_det:,} detections)")
    print(f"   Cameras covered: {sorted(set(k[1] for k in pred_data))}")
else:
    print("\n⚠️  No bbox data in tracklets.json.")
    print("   MOT metrics need bboxes. Only ReID metrics will run.")
    print("   To fix: re-run Cell 6 of gp-stage1-deepocsort, then Stage 5.")

print("=" * 60)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 6: IDF1 / MOTA / MOTP  (motmetrics)  +  HOTA (trackeval)
#
# motmetrics imports scipy which imports numpy.  The current kernel's
# sys.modules still holds the OLD numpy from session start, so the
# direct import fails with the _center ufunc error in some envs.
#
# Workaround: run the motmetrics evaluation in a fresh subprocess
# started AFTER Cell 1 installed numpy 2.1.x.  The subprocess sees
# the new numpy on disk and imports cleanly.
# ══════════════════════════════════════════════════════════════════
import subprocess, sys, json as _json
import numpy as np
import pandas as pd

mot_results = {}   # safe default so Cell 9 summary never hits NameError

if not has_bboxes or not pred_data:
    print("⚠️  Skipping MOT metrics — no bbox data.")
    print("   Re-run Stage 1-B Cell 6 (updated version) to enable these.")
else:
    print("=" * 60)
    print("MOT EVALUATION: IDF1 / MOTA / MOTP  (subprocess)")
    print("=" * 60)

    # ── Run motmetrics in-process (no subprocess) ────────────────────────────────
    print("  Running motmetrics in-process …")
    # numpy 2.x compat: np.asfarray was removed in NumPy 2.0
    if not hasattr(np, 'asfarray'):
        np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)
    per_cam_results = {}
    try:
        import motmetrics as mm
        gt_keys   = set(gt_mot.keys())
        pred_keys = set(pred_data.keys())
        eval_keys = (gt_keys & pred_keys) if gt_keys else pred_keys

        accumulators, names = [], []
        for cam_key in sorted(eval_keys):
            scenario, cam = cam_key
            gt_fc   = gt_mot.get(cam_key, {})
            pred_fc = pred_data[cam_key]
            acc     = mm.MOTAccumulator(auto_id=False)
            for frame in sorted(set(gt_fc) | set(pred_fc)):
                gt_o   = gt_fc.get(frame, [])
                pred_o = pred_fc.get(frame, [])
                gt_ids   = [str(o[0]) for o in gt_o]
                pred_ids = [str(o[0]) for o in pred_o]
                if gt_o and pred_o:
                    gt_b   = np.array([[o[1],o[2],o[1]+o[3],o[2]+o[4]] for o in gt_o])
                    pred_b = np.array([[o[1],o[2],o[1]+o[3],o[2]+o[4]] for o in pred_o])
                    dist   = mm.distances.iou_matrix(gt_b, pred_b, max_iou=0.5)
                elif gt_o:
                    dist = np.empty((len(gt_ids), 0))
                elif pred_o:
                    dist = np.empty((0, len(pred_ids)))
                else:
                    continue
                acc.update(gt_ids, pred_ids, dist, frameid=frame)
            accumulators.append(acc)
            names.append(f"{scenario}/{cam}")

        mh      = mm.metrics.create()
        summary = mh.compute_many(
            accumulators,
            metrics=["idf1","mota","motp","num_switches",
                     "num_false_positives","num_misses","precision","recall"],
            names=names, generate_overall=True,
        )
        overall = summary.loc["OVERALL"] if "OVERALL" in summary.index else summary.iloc[-1]
        mot_results = {
            "idf1":            float(overall["idf1"]),
            "mota":            float(overall["mota"]),
            "motp":            float(overall["motp"]),
            "precision":       float(overall["precision"]),
            "recall":          float(overall["recall"]),
            "id_switches":     int(overall["num_switches"]),
            "false_positives": int(overall["num_false_positives"]),
            "false_negatives": int(overall["num_misses"]),
        }
        per_cam_results = {
            name: {"idf1": float(summary.loc[name]["idf1"]),
                   "mota": float(summary.loc[name]["mota"])}
            for name in names
        }
    except Exception as _mot_exc:
        import traceback as _tb
        _mot_err_txt = _tb.format_exc()
        print(f"⚠️  motmetrics failed: {_mot_exc}")
        print(_mot_err_txt)
        try:
            with open('/kaggle/working/motmetrics_error.txt', 'w') as _ef:
                _ef.write(_mot_err_txt)
        except Exception:
            pass
        print("   MOT metrics (IDF1/MOTA/MOTP) will be skipped.")

    if mot_results:
        gt_lbl = "Official AIC22 GT" if gt_mot else "Proxy (no official GT)"
        print(f"\n{'='*60}")
        print(f"MOT METRICS — OVERALL  [{gt_lbl}]")
        print(f"{'='*60}")
        for k, label in [
            ("idf1",     "IDF1  (primary AIC22 metric)"),
            ("mota",     "MOTA"),
            ("motp",     "MOTP"),
            ("precision","Precision"),
            ("recall",   "Recall"),
        ]:
            print(f"  {label:<35} {mot_results[k]*100:.2f}%")
        print(f"  {'ID Switches':<35} {mot_results['id_switches']}")
        print(f"  {'False Positives':<35} {mot_results['false_positives']}")
        print(f"  {'False Negatives':<35} {mot_results['false_negatives']}")

        if per_cam_results:
            print("\nPer-camera IDF1:")
            for cam, cam_r in sorted(per_cam_results.items()):
                print(f"  {cam:<20}  IDF1={cam_r['idf1']*100:5.1f}%"
                      f"  MOTA={cam_r['mota']*100:5.1f}%")

    # ── HOTA via trackeval ─────────────────────────────────────────
    print("\nRunning HOTA (trackeval) …")
    try:
        import trackeval

        seqmap_path = EVAL_DIR / "seqmap.txt"
        with open(seqmap_path, "w") as f:
            f.write("name\n")
            for cam_key in sorted(pred_data.keys()):
                f.write(f"{cam_key[0]}_{cam_key[1]}\n")

        if gt_mot:
            for (scenario, camera), frame_dict in gt_mot.items():
                seq_name = f"{scenario}_{camera}"
                seq_dir  = GT_DIR / seq_name / "gt"
                seq_dir.mkdir(parents=True, exist_ok=True)
                with open(seq_dir / "gt.txt", "w") as f:
                    for frame, objs in sorted(frame_dict.items()):
                        for tid, x, y, w, h in objs:
                                if tid < 0:
                                    continue  # skip ignore regions
                                f.write(f"{frame},{tid},{x:.1f},{y:.1f},"
                                        f"{w:.1f},{h:.1f},1,1,-1,-1\n")
                with open(GT_DIR / seq_name / "seqinfo.ini", "w") as f:
                    _pred_fr = set(pred_data.get((scenario, camera), {}).keys())
                    _all_fr  = set(frame_dict.keys()) | _pred_fr
                    seq_len  = max(_all_fr) if _all_fr else 1
                    f.write(f"[Sequence]\nname={seq_name}\n"
                            f"framerate=30\nseqLength={seq_len}\n")

            TE_PRED = EVAL_DIR / "trackers" / "stage4" / "data"
            TE_PRED.mkdir(parents=True, exist_ok=True)
            for (scenario, camera), frame_dict in pred_data.items():
                with open(TE_PRED / f"{scenario}_{camera}.txt", "w") as f:
                    for frame in sorted(frame_dict.keys()):
                        for gid, x, y, w, h, conf in frame_dict[frame]:
                            f.write(f"{frame},{gid},{x:.1f},{y:.1f},"
                                    f"{w:.1f},{h:.1f},{conf:.4f},-1,-1,-1\n")

            evaluator = trackeval.Evaluator({
                "USE_PARALLEL": False, "NUM_PARALLEL_CORES": 1,
                "PRINT_RESULTS": False,
            })
            dataset = trackeval.datasets.MotChallenge2DBox({
                "GT_FOLDER":        str(GT_DIR),
                "TRACKERS_FOLDER":  str(EVAL_DIR / "trackers"),
                "TRACKERS_TO_EVAL": ["stage4"],
                "SEQMAP_FILE":      str(seqmap_path),
                "SPLIT_TO_EVAL":    "train",
                "INPUT_AS_ZIP":     False,
                "SKIP_SPLIT_FOL":   True,
                "PRINT_CONFIG":     False,
            })
            te_metrics = [trackeval.metrics.HOTA(),
                          trackeval.metrics.CLEAR(),
                          trackeval.metrics.Identity()]
            te_results, _ = evaluator.evaluate([dataset], te_metrics)
            _cseq = te_results["MotChallenge2DBox"]["stage4"]["COMBINED_SEQ"]
            _cls_key = "pedestrian" if "pedestrian" in _cseq else next(iter(_cseq))
            hota_score = _cseq[_cls_key]["HOTA"]["HOTA"].mean()
            print(f"  HOTA ({_cls_key}): {hota_score*100:.2f}%")
            mot_results["hota"] = float(hota_score)
        else:
            print("  ℹ️  HOTA requires official gt.txt — attach AIC22 dataset.")
    except Exception as e:
        import traceback as _tb2
        _hota_err = _tb2.format_exc()
        print(f"  ⚠️  HOTA failed: {e}")
        print(_hota_err)
        try:
            with open('/kaggle/working/hota_error.txt', 'w') as _hef:
                _hef.write(_hota_err)
        except Exception:
            pass

print("=" * 60)


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 7: REID EVALUATION — mAP / CMC@1 / CMC@5 / CMC@10
#   GPU-accelerated: precomputes the full N×N cosine-similarity
#   matrix in one batched matmul on CUDA (falls back to CPU).
# ══════════════════════════════════════════════════════════════════
import torch
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm
import time

# average_precision_score: prefer sklearn; if scipy import still broken,
# fall back to an equivalent pure-numpy implementation.
try:
    from sklearn.metrics import average_precision_score as _sklearn_ap
    def average_precision_score(y_true, y_score):
        return float(_sklearn_ap(y_true, y_score))
    print("AP backend: sklearn")
except Exception as _sk_err:
    print(f"sklearn unavailable ({_sk_err}); using numpy AP fallback")
    def average_precision_score(y_true, y_score):
        """Pure-numpy AP: sum(precision_at_k * relevance_at_k) / n_pos."""
        order    = np.argsort(-np.asarray(y_score, dtype=np.float32))
        y_sorted = np.asarray(y_true, dtype=np.float32)[order]
        tp       = np.cumsum(y_sorted)
        fp       = np.cumsum(1.0 - y_sorted)
        prec     = tp / (tp + fp)
        n_pos    = float(y_sorted.sum())
        return float(np.sum(prec * y_sorted) / max(n_pos, 1.0))

print("=" * 60)
print("CELL 7: RE-ID EVALUATION: mAP / CMC")
print(f"        Device: {DEVICE}")
print("=" * 60)

reid_results = {
    "map": 0.0, "cmc_at_1": 0.0, "cmc_at_5": 0.0,
    "cmc_at_10": 0.0, "cmc_at_20": 0.0,
    "n_queries": 0, "gt_type": "none",
}

if features is None or N == 0:
    print("⚠️  No ReID features found — skipping mAP/CMC.")
    print("   Attach ali369/gp-stage-2 as an input to enable ReID metrics.")
else:
    # ── Build query / GT ─────────────────────────────────────────
    tracklet_to_idx = {m["tracklet_id"]: i for i, m in enumerate(reid_meta)}

    has_reid_gt   = bool(gt_reid)
    query_indices = []
    gt_matrix     = {}   # query_idx → [positive gallery indices]

    if has_reid_gt:
        print("GT source : AIC22 train-tracks.json (official)")
        from pathlib import Path as _P
        stem_to_idx = {}
        for i, m in enumerate(reid_meta):
            for cp in m.get("crop_paths", []):
                stem_to_idx[_P(cp).stem] = i

        vehicle_to_tidxs = defaultdict(list)
        for vid, crops in gt_reid.items():
            for cp in crops:
                stem = _P(cp).stem
                if stem in stem_to_idx:
                    vehicle_to_tidxs[vid].append(stem_to_idx[stem])

        multi_veh = {v: list(set(idxs)) for v, idxs in vehicle_to_tidxs.items()
                     if len(set(idxs)) > 1}
        print(f"  Vehicles with >=2 tracklets matched: {len(multi_veh)}")

        for vid, idxs in list(multi_veh.items())[:500]:
            q_idx = idxs[0]
            query_indices.append(q_idx)
            gt_matrix[q_idx] = idxs[1:]
    else:
        print("GT source : camera-consistency proxy (same scenario, diff camera)")
        for q_idx, m in enumerate(reid_meta[:400]):
            positives = [
                i for i, mm_ in enumerate(reid_meta)
                if mm_.get("scenario") == m.get("scenario")
                and mm_.get("camera_id") != m.get("camera_id")
                and i != q_idx
            ]
            if positives:
                query_indices.append(q_idx)
                gt_matrix[q_idx] = positives

    print(f"  Queries : {len(query_indices)}  |  Gallery : {N}")

    # ── GPU: precompute full N×N similarity matrix in chunks ─────
    print(f"\nBuilding {N}x{N} similarity matrix on {DEVICE} …")
    t0 = time.time()

    CHUNK = 4096   # rows processed per GPU kernel call
    sim_np = np.empty((N, N), dtype=np.float32)

    feat_t = torch.from_numpy(features).to(DEVICE)   # (N, D) fp32

    with torch.no_grad():
        for start in tqdm(range(0, N, CHUNK), desc="GPU sim-matrix"):
            end  = min(start + CHUNK, N)
            rows = feat_t[start:end]          # (chunk, D)
            sims = torch.mm(rows, feat_t.T)   # (chunk, N)
            sim_np[start:end] = sims.cpu().numpy()

    # Mask self-similarity so each row's own entry is never retrieved
    np.fill_diagonal(sim_np, -1.0)

    elapsed = time.time() - t0
    device_label = "GPU" if DEVICE.type == "cuda" else "CPU"
    print(f"  Similarity matrix built in {elapsed:.1f}s  [{device_label}]")

    # ── Evaluate AP / CMC ────────────────────────────────────────
    aps      = []
    cmc_hits = np.zeros(N)

    for q_idx in tqdm(query_indices, desc="Computing AP / CMC", unit="q"):
        positives = gt_matrix.get(q_idx, [])
        if not positives:
            continue
        sims_q   = sim_np[q_idx]
        sims_q   = np.nan_to_num(sims_q, nan=0.0, posinf=1.0, neginf=-1.0)
        gt_flags = np.zeros(N)
        for p in positives:
            if p < N:
                gt_flags[p] = 1.0
        if gt_flags.sum() == 0:
            continue
        aps.append(average_precision_score(gt_flags, sims_q))
        ranking   = np.argsort(-sims_q)
        first_hit = np.where(gt_flags[ranking] == 1)[0]
        if len(first_hit) > 0:
            cmc_hits[first_hit[0]] += 1

    n_q  = len(aps) or 1
    cmc  = np.cumsum(cmc_hits) / n_q
    gt_lbl = "AIC22 Official GT" if has_reid_gt else "Camera-Proxy GT"

    print(f"\n{'='*60}")
    print(f"RE-ID METRICS  [{gt_lbl}]  (n={n_q} queries)")
    print(f"{'='*60}")
    print(f"  mAP    : {np.mean(aps)*100:.2f}%")
    print(f"  CMC@1  : {cmc[0]*100:.2f}%")
    print(f"  CMC@5  : {cmc[4]*100:.2f}%"  if len(cmc) >  4 else "  CMC@5  : n/a")
    print(f"  CMC@10 : {cmc[9]*100:.2f}%"  if len(cmc) >  9 else "  CMC@10 : n/a")
    print(f"  CMC@20 : {cmc[19]*100:.2f}%" if len(cmc) > 19 else "  CMC@20 : n/a")
    print(f"{'='*60}")

    reid_results = {
        "map":       float(np.mean(aps)) if aps else 0.0,
        "cmc_at_1":  float(cmc[0]),
        "cmc_at_5":  float(cmc[4])  if len(cmc) >  4 else 0.0,
        "cmc_at_10": float(cmc[9])  if len(cmc) >  9 else 0.0,
        "cmc_at_20": float(cmc[19]) if len(cmc) > 19 else 0.0,
        "n_queries": n_q,
        "gt_type":   "official" if has_reid_gt else "proxy",
    }

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 8: VISUAL SAMPLES
#   - Crop strips of top multi-camera global tracks
#   - Embeds Stage 4 demo videos (if they exist in the zip)
# ══════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display, HTML, Image as IPImage
import base64

COLOURS_HEX = ["#00DC00","#0088FF","#FF3232","#AA00AA",
               "#00DDDD","#B46400","#0064FF","#64C864"]

import re as _re_vis

# -- Stage 1B zip handle (opened once, reused across all load_crop calls) -----
_S1B_ZIP_PATH = Path('/kaggle/input/notebooks/gumfreddy/gp-stage-1b-detection-tracking-deepocsort/_output_.zip')
_s1b_zf       = None   # zipfile.ZipFile handle
_s1b_names    = set()  # set of entry names in the zip

def _open_s1b_zip():
    global _s1b_zf, _s1b_names
    if _s1b_zf is None and _S1B_ZIP_PATH.exists():
        import zipfile as _zfmod
        _s1b_zf    = _zfmod.ZipFile(_S1B_ZIP_PATH)
        _s1b_names = set(_s1b_zf.namelist())
        print(f"  [viz] Opened Stage-1B zip: {len(_s1b_names)} entries")

def load_crop(path_str, size=(128, 192)):
    import io as _io
    clean = path_str.replace("\\", "/")

    # -- 1. Try the pre-saved crop file (CROPS_ROOT) ------------------
    p = Path(clean)
    if not p.is_absolute():
        p = CROPS_ROOT / clean
    if p.exists():
        try:
            return PILImage.open(p).convert("RGB").resize(size)
        except Exception:
            pass

    # -- 2. Read directly from Stage 1B zip --------------------------
    _open_s1b_zip()
    if _s1b_zf is not None:
        try:
            # Stage 1B stores crops at 'crops/{path}' inside the zip
            for zpath in (f"crops/{clean}", clean):
                if zpath in _s1b_names:
                    return (PILImage.open(_io.BytesIO(_s1b_zf.read(zpath)))
                                    .convert("RGB").resize(size))
        except Exception:
            pass

    # -- 3. Fallback: re-crop from raw AIC22 frame (if img1 exists) --
    if AIC22_ROOT is not None:
        try:
            m = _re_vis.match(
                r'(train|validation)/\w+/(S\d+)_(c\d+)_(track_\d+)_frame_(\d+)\.jpg$',
                clean)
            if m:
                split, scenario, camera, tracklet_id, frame_str = m.groups()
                frame_no = int(frame_str)
                tdata = stage1_tracklets.get(tracklet_id, {})
                bbox  = next((b for b in tdata.get("bboxes", [])
                              if b["frame"] == frame_no), None)
                if bbox is not None:
                    for fn in (f"{frame_no + 1:06d}.jpg", f"{frame_no:06d}.jpg"):
                        frame_file = AIC22_ROOT / split / scenario / camera / "img1" / fn
                        if frame_file.exists():
                            img = PILImage.open(frame_file).convert("RGB")
                            x1 = max(0, int(bbox["x1"]))
                            y1 = max(0, int(bbox["y1"]))
                            x2 = min(img.width,  x1 + int(bbox["w"]))
                            y2 = min(img.height, y1 + int(bbox["h"]))
                            if x2 > x1 and y2 > y1:
                                return img.crop((x1, y1, x2, y2)).resize(size)
                            break
        except Exception:
            pass

    return PILImage.new("RGB", size, color=(40, 40, 40))

def show_global_track_strip(gid, cam_tracks, n_per_cam=8):
    """Show a full tracklet strip per camera for a global vehicle ID.
    Each row = one camera's DeepOCSORT tracklet (sampled evenly across all frames).
    n_per_cam=8 gives a wide temporal view of each tracklet.
    Label shows: scenario / camera / track_id / (frame count).
    Frame number printed under each crop so you can see the temporal spread.
    """
    n_cams = len(cam_tracks)
    fig, axes = plt.subplots(n_cams, n_per_cam + 1,
                             figsize=(2.5 * (n_per_cam + 1), 2.0 * n_cams))
    if n_cams == 1:
        axes = [axes]
    for idx, ct in enumerate(cam_tracks):
        cam_id   = ct.get("camera_id", "?")
        scenario = ct.get("scenario",  "?")
        track_id = ct.get("track_id",  "?")   # DeepOCSORT local track id
        frames_  = ct.get("frames", [])
        colour   = COLOURS_HEX[idx % len(COLOURS_HEX)]
        ax_lbl   = axes[idx][0]
        ax_lbl.axis("off")
        n_fr = len(frames_)
        ax_lbl.text(0.5, 0.5, f"{scenario}\n{cam_id}\n{track_id}\n({n_fr} fr)",
                    ha="center", va="center", fontsize=9,
                    color="white", fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.4",
                              facecolor=colour, alpha=0.85),
                    transform=ax_lbl.transAxes)
        sampled = ([frames_[i]
                    for i in np.linspace(0, len(frames_) - 1,
                    min(n_per_cam, len(frames_)), dtype=int)]
                   if frames_ else [])
        for col, fpath in enumerate(sampled, start=1):
            ax = axes[idx][col]
            ax.imshow(load_crop(fpath))
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_visible(False)
            m_fr = _re_vis.search(r'frame_(\d+)', fpath)
            if m_fr:
                ax.set_xlabel(f"fr{int(m_fr.group(1))}", fontsize=6, labelpad=1)
        for col in range(len(sampled) + 1, n_per_cam + 1):
            axes[idx][col].axis("off")
    fig.suptitle(f"Global ID {gid}  —  {n_cams} cameras",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    sp = RESULTS_DIR / f"visual_global_{gid:05d}.png"
    plt.savefig(sp, dpi=100, bbox_inches="tight")
    plt.close()
    display(IPImage(str(sp)))

# ── Top multi-camera tracks ───────────────────────────────────
multi_tracks = [
    (int(gid), tracks)
    for gid, tracks in global_tracklets.items()
    if isinstance(tracks, list) and len(tracks) > 1
]
multi_tracks.sort(
    key=lambda x: (len(x[1]),
                   sum(len(t.get("frames", [])) for t in x[1])),
    reverse=True
)

print(f"Multi-camera global tracks: {len(multi_tracks)}")
print("Sorted by: (n_cameras DESC, total_frames DESC)  -->  most-covered vehicles first")
print("  Each row = one camera's DeepOCSORT tracklet.  8 evenly-sampled frames per row.")
print("  Frame number shown under each crop.  Track ID shown in row label.")
print(f"Visualising top 50 ...\n")
for gid, tracks in multi_tracks[:50]:
    show_global_track_strip(gid, tracks)

# ── Embed Stage 4 demo videos ─────────────────────────────────
video_dir = EXTRACT_ROOT / "stage4" / "demo_videos"
if not video_dir.exists():
    video_dir = WORK / "stage4" / "demo_videos"
demo_vids = sorted(video_dir.glob("*.mp4"))[:3] if video_dir.exists() else []

if demo_vids:
    print(f"\n🎬 Embedding {len(demo_vids)} Stage 4 demo videos …")
    for vp in demo_vids:
        b64 = base64.b64encode(vp.read_bytes()).decode()
        display(HTML(
            f'<p><b>{vp.name}</b></p>'
            f'<video width="700" controls autoplay loop>'
            f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
            f'</video>'
        ))
else:
    print("(No Stage 4 demo videos found — run Stage 4 Cell 10 first)")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 9: RESULTS SUMMARY + PLOTS + JSON SAVE
# ══════════════════════════════════════════════════════════════════
import json, shutil
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from IPython.display import display, Image as IPImage

print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)

print(f"\n{'Metric':<30} {'Value':>10}  Source")
print("-" * 60)

if mot_results:
    for k, label, src in [
        ("idf1",           "IDF1  (AIC22 primary)",  "motmetrics"),
        ("hota",           "HOTA",                   "trackeval"),
        ("mota",           "MOTA",                   "motmetrics"),
        ("motp",           "MOTP",                   "motmetrics"),
        ("precision",      "Precision",              "motmetrics"),
        ("recall",         "Recall",                 "motmetrics"),
        ("id_switches",    "ID Switches",            "motmetrics"),
    ]:
        if k not in mot_results:
            continue
        v = mot_results[k]
        disp = f"{v*100:.2f}%" if k not in ("id_switches",) else str(int(v))
        print(f"  {label:<28} {disp:>10}  {src}")
else:
    print("  MOT metrics skipped (no bbox data)")

print()
for k, label in [
    ("map",       "mAP"),
    ("cmc_at_1",  "CMC@1"),
    ("cmc_at_5",  "CMC@5"),
    ("cmc_at_10", "CMC@10"),
]:
    v = reid_results.get(k, 0.0)
    print(f"  {label:<28} {v*100:>10.2f}%  ReID [{reid_results['gt_type']}]")

print("=" * 60)

# ── Bar chart ─────────────────────────────────────────────────
metric_vals = {}
for k, lbl in [("idf1","IDF1"),("hota","HOTA"),("mota","MOTA")]:
    if k in mot_results:
        metric_vals[lbl] = mot_results[k] * 100
for k, lbl in [("map","mAP"),("cmc_at_1","CMC@1"),("cmc_at_5","CMC@5")]:
    metric_vals[lbl] = reid_results.get(k, 0.0) * 100

summary_plot = None
if metric_vals:
    keys   = list(metric_vals.keys())
    vals   = [metric_vals[k] for k in keys]
    n_mot  = sum(1 for k in keys if k in ("IDF1","HOTA","MOTA"))
    cols   = (["#4C9BE8"] * n_mot + ["#E87C4C"] * (len(keys) - n_mot))
    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(keys, vals, color=cols, edgecolor="black", alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{v:.1f}%", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
    ax.set_ylabel("Score (%)", fontsize=11)
    ax.set_title("AIC22 Track 1 — Stage 5 Evaluation Results",
                 fontsize=13, fontweight="bold")
    ax.set_ylim(0, min(110, max(vals or [10]) + 20))
    ax.grid(axis="y", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    summary_plot = RESULTS_DIR / "evaluation_summary.png"
    plt.tight_layout()
    plt.savefig(summary_plot, dpi=130)
    plt.close()
    display(IPImage(str(summary_plot)))

# ── Save JSON ─────────────────────────────────────────────────
all_results = {
    "timestamp":        datetime.now(timezone.utc).isoformat(),
    "gt_mode":          GT_MODE,
    "tracking_metrics": mot_results,
    **({"mot_error": open('/kaggle/working/motmetrics_error.txt').read()} if __import__('pathlib').Path('/kaggle/working/motmetrics_error.txt').exists() else {}),
    "reid_metrics":     reid_results,
    "pipeline_stats": {
        "global_ids":       len(global_tracklets),
        "multi_cam_tracks": len([v for v in global_tracklets.values()
                                  if isinstance(v, list) and len(v) > 1]),
        "feature_dim":      int(D),
        "n_tracklets":      N,
    },
}

results_json = RESULTS_DIR / "evaluation_results.json"
with open(results_json, "w") as f:
    json.dump(all_results, f, indent=2)

shutil.copy(results_json, WORK / "evaluation_results.json")
if summary_plot and summary_plot.exists():
    shutil.copy(summary_plot, WORK / "evaluation_summary.png")

print(f"\n✅ Results saved to /kaggle/working/evaluation_results.json")
print(json.dumps(all_results, indent=2))


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 10: PUSH RESULTS TO KAGGLE DATASET
# Supports both old-style API keys (username + key) and new
# KGAT_ token format.  Falls back gracefully if push fails.
# ══════════════════════════════════════════════════════════════════
import os, json, subprocess, shutil, requests
from pathlib import Path
from datetime import datetime, timezone

print("=" * 60)
print("CELL 10: KAGGLE PUSH")
print("=" * 60)

token = os.environ.get("KAGGLE_API_TOKEN", "")

def _log(msg):
    ts = datetime.now(timezone.utc).strftime("%H:%M:%S")
    print(f"[{ts}] {msg}")

if not token:
    _log("⚠️  KAGGLE_API_TOKEN not found — skipping push.")
    _log("   Your results are in this notebook's Output tab.")
    _log("   To enable: Notebook Settings → Secrets → KAGGLE_API_TOKEN")
else:
    # ── Write kaggle.json in the format the CLI expects ───────────
    kaggle_cfg = Path("/root/.kaggle/kaggle.json")
    kaggle_cfg.parent.mkdir(exist_ok=True)

    is_kgat = token.startswith("KGAT_")

    if is_kgat:
        # New token format — CLI >= 1.6 accepts {"token": "KGAT_..."}
        kaggle_cfg.write_text(json.dumps({"token": token}))
        _log("Token type: KGAT_  (new API token)")
    else:
        # Old token format — split on first | to get username:key
        # Classic kaggle.json stored as "username|key" or just the key
        if "|" in token:
            uname_part, key_part = token.split("|", 1)
            kaggle_cfg.write_text(json.dumps({
                "username": uname_part.strip(),
                "key":      key_part.strip(),
            }))
            _log(f"Token type: classic  (username={uname_part.strip()})")
        else:
            # Treat whole string as API key with username from env
            uname_part = os.environ.get("KAGGLE_USERNAME", KAGGLE_USERNAME)
            kaggle_cfg.write_text(json.dumps({
                "username": uname_part,
                "key":      token.strip(),
            }))
            _log(f"Token type: classic key  (username={uname_part})")

    kaggle_cfg.chmod(0o600)

    # ── Resolve kaggle username via the API ───────────────────────
    uname = KAGGLE_USERNAME   # fallback
    try:
        if is_kgat:
            # KGAT tokens use HTTP Basic Auth with empty username
            r = requests.get(
                "https://www.kaggle.com/api/v1/users/me",
                auth=("", token),
                timeout=10,
            )
        else:
            # Classic key — read from kaggle.json
            cfg = json.loads(kaggle_cfg.read_text())
            r = requests.get(
                "https://www.kaggle.com/api/v1/users/me",
                auth=(cfg.get("username",""), cfg.get("key", token)),
                timeout=10,
            )
        if r.status_code == 200:
            uname = r.json().get("userName", uname)
            _log(f"Authenticated as   : {uname}")
        else:
            _log(f"Auth check HTTP {r.status_code}  — using fallback username: {uname}")
    except Exception as e:
        _log(f"Auth check failed ({e}) — using fallback username: {uname}")

    # If it was a classic key, update kaggle.json with confirmed username
    if not is_kgat:
        kaggle_cfg.write_text(json.dumps({"username": uname, "key": token.strip()}))
        kaggle_cfg.chmod(0o600)

    # ── Assemble dataset directory ─────────────────────────────────
    RESULTS_SLUG = "gp-evaluation-results"
    DS_DIR       = WORK / "kaggle_push"
    DS_DIR.mkdir(exist_ok=True)

    copied = []
    for fname in [
        "evaluation_results.json",
        "evaluation_summary.png",
        "ablation_study.png",
        "similarity_distribution.png",
    ]:
        src = WORK / fname
        if src.exists():
            shutil.copy(src, DS_DIR / fname)
            copied.append(fname)

    for img in sorted(RESULTS_DIR.glob("visual_global_*.png"))[:10]:
        shutil.copy(img, DS_DIR / img.name)
        copied.append(img.name)

    _log(f"Files staged for push ({len(copied)}): {copied}")

    ts_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    (DS_DIR / "dataset-metadata.json").write_text(json.dumps({
        "title":    f"GP Evaluation Results AIC22 {ts_str}",
        "id":       f"{uname}/{RESULTS_SLUG}",
        "licenses": [{"name": "CC0-1.0"}],
    }, indent=2))

    # ── Push via kaggle CLI ────────────────────────────────────────
    def run_kaggle(args):
        result = subprocess.run(
            ["kaggle"] + args + ["--quiet"],
            capture_output=True, text=True,
        )
        return result

    _log("Attempting dataset create …")
    r1 = run_kaggle(["datasets", "create", "-p", str(DS_DIR)])
    if r1.returncode == 0:
        _log(f"Dataset created successfully!")
        _log(f"   URL → https://www.kaggle.com/datasets/{uname}/{RESULTS_SLUG}")
    else:
        _log(f"Create failed (probably exists already).  Trying version push …")
        _log(f"   create stderr: {r1.stderr[:300]}")
        commit_msg = f"Eval results {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC"
        r2 = run_kaggle([
            "datasets", "version",
            "-p", str(DS_DIR),
            "-m", commit_msg,
        ])
        if r2.returncode == 0:
            _log(f"Dataset version pushed!")
            _log(f"   URL → https://www.kaggle.com/datasets/{uname}/{RESULTS_SLUG}")
        else:
            _log(f"Version push also failed.")
            _log(f"   version stderr: {r2.stderr[:500]}")
            _log("")
            _log("Your results are available in this notebook's Output tab.")
            _log("Download 'evaluation_results.json' from there.")

print("")
print("=" * 60)
print(f"✅ Stage 5 complete.  Total wall time: {time.time()-WALL_START:.0f}s")
print("=" * 60)